In [1]:
!pip install "dacite~=1.6.0" mlflow dagshub scikit-learn -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.2/49.2 kB 1.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 2.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 66.5 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.7 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 39.8 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 273.1/273.1 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.2/68.2 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 208.4/208.4 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.2/132.2 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

import mlflow
import mlflow.sklearn
import dagshub
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler

RANDOM_STATE = 42
REPO_OWNER = 'ejoba22'
REPO_NAME = 'IEEE-CIS-Fraud-Detection'
EXPERIMENT_NAME = 'LogisticRegression_Training'

dagshub.init(repo_owner=REPO_OWNER, repo_name=REPO_NAME, mlflow=True)

if mlflow.active_run() is not None:
    mlflow.end_run()

experiment = mlflow.set_experiment(EXPERIMENT_NAME)
EXPERIMENT_ID = experiment.experiment_id

print('MLflow tracking URI:', mlflow.get_tracking_uri())
print('MLflow experiment:', experiment.name)
print('MLflow experiment id:', EXPERIMENT_ID)

❗❗❗ AUTHORIZATION REQUIRED ❗❗❗

Output()



Open the following link in your browser to authorize the client:
https://dagshub.com/login/oauth/authorize?state=9d8e4076-da75-485c-bacd-036d6249cf82&client_id=32b60ba385aa7cecf24046d8195a71c07dd345d9657977863b52e7748e0f0f28&middleman_request_id=4b1fb54bb8ef9d9965845cda569cc8b945a4847f8e73516a1837de8631e76781




Accessing as ejoba22

Initialized MLflow to track repo "ejoba22/IEEE-CIS-Fraud-Detection"

Repository ejoba22/IEEE-CIS-Fraud-Detection initialized!

2026/05/03 17:38:23 INFO mlflow.tracking.fluent: Experiment with name 'LogisticRegression_Training' does not exist. Creating a new experiment.


MLflow tracking URI: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow
MLflow experiment: LogisticRegression_Training
MLflow experiment id: 1


In [5]:
def reduce_mem_usage(df):
    start_mem = df.memory_usage(deep=True).sum() / 1024**2

    for col in df.columns:
        col_type = df[col].dtype
        if col_type == object:
            continue

        c_min = df[col].min()
        c_max = df[col].max()

        if str(col_type).startswith('int'):
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        elif str(col_type).startswith('float'):
            df[col] = df[col].astype(np.float32)

    end_mem = df.memory_usage(deep=True).sum() / 1024**2
    print(f'Memory reduced from {start_mem:.1f} MB to {end_mem:.1f} MB')
    return df


from pathlib import Path

DATA_DIR_CANDIDATES = [
    Path('/kaggle/input/competitions/ieee-fraud-detection'),
    Path('/kaggle/input/ieee-fraud-detection')
]

DATA_DIR = next(
    (path for path in DATA_DIR_CANDIDATES if (path / 'train_transaction.csv').exists()),
    None
)

if DATA_DIR is None:
    raise FileNotFoundError(
        'Could not find IEEE-CIS data. In Kaggle, add the competition dataset to this notebook. '
        f'Tried: {[str(path) for path in DATA_DIR_CANDIDATES]}'
    )

print('Using data directory:', DATA_DIR)

train_transaction = reduce_mem_usage(pd.read_csv(DATA_DIR / 'train_transaction.csv'))
train_identity = reduce_mem_usage(pd.read_csv(DATA_DIR / 'train_identity.csv'))

train = train_transaction.merge(train_identity, on='TransactionID', how='left')
train = reduce_mem_usage(train)
print('Raw shape:', train.shape)

Using data directory: /kaggle/input/competitions/ieee-fraud-detection
Memory reduced from 2062.1 MB to 1203.2 MB
Memory reduced from 143.1 MB to 129.9 MB
Memory reduced from 1603.3 MB to 1603.3 MB
Raw shape: (590540, 434)


# Cleaning


In [6]:
train = train.sort_values('TransactionDT').reset_index(drop=True)
split_idx = int(len(train) * 0.8)
df_tr_raw = train.iloc[:split_idx].copy()
df_val_raw = train.iloc[split_idx:].copy()

df_tr = df_tr_raw.copy()
df_val = df_val_raw.copy()

print(f'Train: {len(df_tr):,} rows  |  Val: {len(df_val):,} rows')

LR_MISSING_THRESHOLD = 0.9
missing = train.isnull().mean()
drop_cols = missing[missing > LR_MISSING_THRESHOLD].index.tolist() + ['TransactionID', 'TransactionDT']

df_tr = df_tr.drop(columns=drop_cols, errors='ignore')
df_val = df_val.drop(columns=drop_cols, errors='ignore')

cat_cols = df_tr.select_dtypes('object').columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    df_tr[col]  = df_tr[col].fillna('missing')
    df_val[col] = df_val[col].fillna('missing')
    le.fit(df_tr[col].astype(str))
    df_tr[col]  = le.transform(df_tr[col].astype(str))
    df_val[col] = df_val[col].astype(str).map(
        lambda x, le=le: le.transform([x])[0] if x in le.classes_ else -1
    )

num_cols = [c for c in df_tr.select_dtypes(include=np.number).columns if c != 'isFraud']
medians  = df_tr[num_cols].median()
df_tr[num_cols]  = df_tr[num_cols].fillna(medians)
df_val[num_cols] = df_val[num_cols].fillna(medians)

with mlflow.start_run(experiment_id=EXPERIMENT_ID, run_name='LogisticRegression_Cleaning'):
    mlflow.log_param('missing_threshold', LR_MISSING_THRESHOLD)
    mlflow.log_param('imputation',        'median')
    mlflow.log_param('cat_encoding',      'label encoding')
    mlflow.log_param('dropped_cols',      len(drop_cols))
    mlflow.log_param('remaining_cols',    df_tr.shape[1])

print(f'Shape after cleaning: {df_tr.shape}')
print(f'Remaining NaNs: {df_tr.isnull().sum().sum()}')

Train: 472,432 rows  |  Val: 118,108 rows
🏃 View run LogisticRegression_Cleaning at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/053d3d6161b04a1598715ff5da8c2006
🧪 View experiment at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
Shape after cleaning: (472432, 420)
Remaining NaNs: 0


# Feature Engineering

In [7]:
for df in [df_tr, df_val]:
    df['TransactionAmt_log'] = np.log1p(df['TransactionAmt'])

with mlflow.start_run(experiment_id=EXPERIMENT_ID, run_name='LogisticRegression_Feature_Engineering'):
    mlflow.log_param('new_features', 'TransactionAmt_log')

print('Feature engineering done.')

🏃 View run LogisticRegression_Feature_Engineering at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/a57dd6c80a3a480e9561b69a313b162f
🧪 View experiment at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
Feature engineering done.


# Feature Selection


In [8]:
X_tr  = df_tr.drop(columns=['isFraud'])
y_tr  = df_tr['isFraud']
X_val = df_val.drop(columns=['isFraud'])
y_val = df_val['isFraud']


low_var = X_tr.var()[X_tr.var() < 0.01].index.tolist()
X_tr  = X_tr.drop(columns=low_var)
X_val = X_val.drop(columns=low_var)


scaler = StandardScaler()
X_tr_scaled  = scaler.fit_transform(X_tr)
X_val_scaled = scaler.transform(X_val)

with mlflow.start_run(experiment_id=EXPERIMENT_ID, run_name='LogisticRegression_Feature_Selection'):
    mlflow.log_param('method',         'drop near-zero variance + StandardScaler')
    mlflow.log_param('low_var_dropped', len(low_var))
    mlflow.log_param('final_features',  X_tr.shape[1])

print(f'Features after selection: {X_tr.shape[1]}')

🏃 View run LogisticRegression_Feature_Selection at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/c5175e0548374f0fba9031748bf1d47c
🧪 View experiment at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
Features after selection: 395


# Training

In [9]:
# Underfit (very strong regularization)
with mlflow.start_run(experiment_id=EXPERIMENT_ID, run_name='LogisticRegression_Training_underfit_strong_regularization'):
    model = LogisticRegression(C=0.0001, max_iter=1000, class_weight='balanced',
                               random_state=RANDOM_STATE)
    model.fit(X_tr_scaled, y_tr)

    train_auc = roc_auc_score(y_tr,  model.predict_proba(X_tr_scaled)[:, 1])
    val_auc   = roc_auc_score(y_val, model.predict_proba(X_val_scaled)[:, 1])

    mlflow.log_param('C',             0.0001)
    mlflow.log_param('class_weight',  'balanced')
    mlflow.log_param('note',          'intentional underfit — C too small')
    mlflow.log_metric('train_auc',    round(train_auc, 4))
    mlflow.log_metric('val_auc',      round(val_auc,   4))
    mlflow.log_metric('overfit_gap',  round(train_auc - val_auc, 4))

    print(f'[Underfit]  train={train_auc:.4f}  val={val_auc:.4f}')

[Underfit]  train=0.8632  val=0.8304
🏃 View run LogisticRegression_Training_underfit_strong_regularization at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/4cbf17d79e704fa8a13982cc92603ecb
🧪 View experiment at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


In [10]:
# Overfit (no regularization)
with mlflow.start_run(experiment_id=EXPERIMENT_ID, run_name='LogisticRegression_Training_weak_regularization_overfit_check'):
    model = LogisticRegression(C=1000, max_iter=1000, class_weight='balanced',
                               random_state=RANDOM_STATE)
    model.fit(X_tr_scaled, y_tr)

    train_auc = roc_auc_score(y_tr,  model.predict_proba(X_tr_scaled)[:, 1])
    val_auc   = roc_auc_score(y_val, model.predict_proba(X_val_scaled)[:, 1])

    mlflow.log_param('C',             1000)
    mlflow.log_param('class_weight',  'balanced')
    mlflow.log_param('note',          'intentional overfit — C too large')
    mlflow.log_metric('train_auc',    round(train_auc, 4))
    mlflow.log_metric('val_auc',      round(val_auc,   4))
    mlflow.log_metric('overfit_gap',  round(train_auc - val_auc, 4))

    print(f'[Overfit]   train={train_auc:.4f}  val={val_auc:.4f}')

[Overfit]   train=0.8738  val=0.8305
🏃 View run LogisticRegression_Training_weak_regularization_overfit_check at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/e3dea245c082404ba6295d70008ff8eb
🧪 View experiment at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1


In [11]:
# Tuned 
best_val_auc = 0
best_C = None
best_gap = None

for C in [0.01, 0.1, 1.0, 10.0]:
    with mlflow.start_run(experiment_id=EXPERIMENT_ID, run_name=f'LogisticRegression_Training_tuned_C{C}'):
        model = LogisticRegression(C=C, max_iter=1000, class_weight='balanced',
                                   random_state=RANDOM_STATE)
        model.fit(X_tr_scaled, y_tr)

        train_auc = roc_auc_score(y_tr,  model.predict_proba(X_tr_scaled)[:, 1])
        val_auc   = roc_auc_score(y_val, model.predict_proba(X_val_scaled)[:, 1])

        mlflow.log_param('C',            C)
        mlflow.log_param('class_weight', 'balanced')
        mlflow.log_metric('train_auc',   round(train_auc, 4))
        mlflow.log_metric('val_auc',     round(val_auc,   4))
        mlflow.log_metric('overfit_gap', round(train_auc - val_auc, 4))

        print(f'[C={C}]  train={train_auc:.4f}  val={val_auc:.4f}')

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_C = C
            best_gap = train_auc - val_auc

print(f'\nBest C: {best_C}  |  Best val AUC: {best_val_auc:.4f}')

[C=0.01]  train=0.8714  val=0.8301
🏃 View run LogisticRegression_Training_tuned_C0.01 at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/66911f9282a54b57af3b21b4ef84fa26
🧪 View experiment at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
[C=0.1]  train=0.8733  val=0.8311
🏃 View run LogisticRegression_Training_tuned_C0.1 at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/c8a01b09e411413ebb307c6d7141ad56
🧪 View experiment at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
[C=1.0]  train=0.8738  val=0.8309
🏃 View run LogisticRegression_Training_tuned_C1.0 at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/df8589b38aad4dc4854a203fb4decdd6
🧪 View experiment at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
[C=10.0]  train=0.8738  val=0.8306
🏃 View run LogisticRegression_Training_tuned_C10.0 at: https://

In [12]:
class LogisticRegressionFraudPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, missing_threshold=0.9, low_variance_threshold=0.01):
        self.missing_threshold = missing_threshold
        self.low_variance_threshold = low_variance_threshold

    def fit(self, X, y=None):
        d = X.copy()
        self.drop_cols_ = d.isnull().mean()[d.isnull().mean() > self.missing_threshold].index.tolist()
        self.drop_cols_ = [col for col in self.drop_cols_ if col != 'isFraud'] + ['TransactionID', 'TransactionDT']

        d = self._basic_transform(d, fit=True)
        self.medians_ = d.median(numeric_only=True)
        d = d.fillna(self.medians_)

        self.low_var_cols_ = d.var()[d.var() < self.low_variance_threshold].index.tolist()
        d = d.drop(columns=self.low_var_cols_, errors='ignore')
        self.feature_columns_ = d.columns.tolist()

        self.scaler_ = StandardScaler()
        self.scaler_.fit(d)
        return self

    def _basic_transform(self, X, fit=False):
        d = X.drop(columns=self.drop_cols_, errors='ignore').copy()
        d = d.drop(columns=['isFraud'], errors='ignore')

        if 'TransactionAmt' in d.columns:
            d['TransactionAmt_log'] = np.log1p(d['TransactionAmt'])

        cat_cols = d.select_dtypes('object').columns.tolist()
        if fit:
            self.category_maps_ = {}
            for col in cat_cols:
                values = d[col].fillna('missing').astype(str)
                self.category_maps_[col] = {value: i for i, value in enumerate(pd.Index(values.unique()))}

        for col in cat_cols:
            values = d[col].fillna('missing').astype(str)
            d[col] = values.map(self.category_maps_[col]).fillna(-1).astype('float32')

        for col in d.select_dtypes('float64').columns:
            d[col] = d[col].astype('float32')
        for col in d.select_dtypes('int64').columns:
            d[col] = d[col].astype('int32')

        return d

    def transform(self, X):
        d = self._basic_transform(X, fit=False)
        d = d.fillna(self.medians_)
        d = d.drop(columns=self.low_var_cols_, errors='ignore')
        d = d.reindex(columns=self.feature_columns_, fill_value=0)
        return self.scaler_.transform(d)


required_final_vars = ['best_C', 'df_tr_raw', 'df_val_raw', 'EXPERIMENT_ID']
missing_final_vars = [name for name in required_final_vars if name not in globals()]
if missing_final_vars:
    raise RuntimeError(f'Run the earlier notebook sections first. Missing variables: {missing_final_vars}')

final_pipeline = Pipeline(steps=[
    ('preprocess', LogisticRegressionFraudPreprocessor(
        missing_threshold=LR_MISSING_THRESHOLD,
        low_variance_threshold=0.01
    )),
    ('model', LogisticRegression(
        C=best_C,
        max_iter=1000,
        class_weight='balanced',
        random_state=RANDOM_STATE
    ))
])

with mlflow.start_run(experiment_id=EXPERIMENT_ID, run_name='LogisticRegression_Final_Pipeline_Register'):
    X_train_raw = df_tr_raw.drop(columns=['isFraud'])
    y_train_raw = df_tr_raw['isFraud']
    X_val_raw = df_val_raw.drop(columns=['isFraud'])
    y_val_raw = df_val_raw['isFraud']

    final_pipeline.fit(X_train_raw, y_train_raw)

    train_auc = roc_auc_score(y_train_raw, final_pipeline.predict_proba(X_train_raw)[:, 1])
    val_auc = roc_auc_score(y_val_raw, final_pipeline.predict_proba(X_val_raw)[:, 1])
    gap = train_auc - val_auc

    mlflow.log_param('C', best_C)
    mlflow.log_param('class_weight', 'balanced')
    mlflow.log_param('missing_threshold', LR_MISSING_THRESHOLD)
    mlflow.log_param('imputation', 'median')
    mlflow.log_param('cat_encoding', 'ordinal mapping with unseen=-1')
    mlflow.log_param('feature_engineering', 'TransactionAmt_log')
    mlflow.log_param('feature_selection', 'low variance threshold 0.01')
    mlflow.log_metric('train_auc', round(train_auc, 4))
    mlflow.log_metric('val_auc', round(val_auc, 4))
    mlflow.log_metric('overfit_gap', round(gap, 4))

    mlflow.sklearn.log_model(
        sk_model=final_pipeline,
        artifact_path='logistic_regression_pipeline',
        registered_model_name='IEEE_CIS_LogisticRegression_Best'
    )

    print(f'Final pipeline — train={train_auc:.4f}  val={val_auc:.4f} gap={gap:.4f}')
    print('Registered to Model Registry as: IEEE_CIS_LogisticRegression_Best')

2026/05/03 18:10:45 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/03 18:10:51 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Successfully registered model 'IEEE_CIS_LogisticRegression_Best'.
2026/05/03 18:11:09 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: IEEE_CIS_LogisticRegression_Best, version 1
Created version '1' of model 'IEEE_CIS_LogisticRegression_Best'.


Final pipeline — train=0.8743  val=0.8317 gap=0.0426
Registered to Model Registry as: IEEE_CIS_LogisticRegression_Best
🏃 View run LogisticRegression_Final_Pipeline_Register at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1/runs/f034667164cc4b668e0f2aecacb84b5c
🧪 View experiment at: https://dagshub.com/ejoba22/IEEE-CIS-Fraud-Detection.mlflow/#/experiments/1
